# STATISTICAL LEARNING FINAL PROJECT
# Beta Distribution
## Raissa Kitenge

In [ ]:
import pandas as pd
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
#  Load feature-engineered data

df = pd.read_csv("/content/drive/MyDrive/feature_engineered_real_estate.csv")
df.head()

In [ ]:
# Select predictors

predictor_cols = [
    "TotalFinishedArea",
    "LivingUnits",
    "SaleYear",
    "SaleMonth",
    "AssrLandUse_APT FOUR",
    "AssrLandUse_CONDOMINIUM",
    "AssrLandUse_MULTI DWLG",
    "AssrLandUse_ONE FAMILY",
    "AssrLandUse_THREE FAMILY",
    "AssrLandUse_TWO FAMILY"
]
# Keep only columns that actually exist
predictor_cols = [col for col in predictor_cols if col in df.columns]

X = df[predictor_cols].copy()
y = df["PriceRatio"].copy()

print("Predictors used:")
print(X.columns.tolist())
print("\nShape:", X.shape)


In [ ]:
# Train/test split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)


In [ ]:
# Standardize only continuous predictors

cont_cols = [col for col in ["TotalFinishedArea", "LivingUnits", "SaleYear", "SaleMonth"] if col in X_train.columns]
bin_cols = [col for col in X_train.columns if col not in cont_cols]

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[cont_cols] = scaler.fit_transform(X_train[cont_cols])
X_test_scaled[cont_cols] = scaler.transform(X_test[cont_cols])

In [ ]:
# Rescale PriceRatio from (0.5, 1.5) to (0, 1)
# Beta likelihood requires values strictly inside (0, 1)

def rescale_to_unit_interval(y, lower=0.5, upper=1.5, eps=1e-4):
    y_scaled = (y - lower) / (upper - lower)
    y_scaled = np.clip(y_scaled, eps, 1 - eps)
    return y_scaled

y_train_beta = rescale_to_unit_interval(y_train)
y_test_beta = rescale_to_unit_interval(y_test)

print("\nRescaled target summary:")
print(pd.Series(y_train_beta).describe())

In [ ]:
# Convert to numpy arrays

# Convert all predictors to float
X_train_scaled = X_train_scaled.astype(float)
X_test_scaled = X_test_scaled.astype(float)

X_train_np = X_train_scaled.to_numpy(dtype=float)
X_test_np = X_test_scaled.to_numpy(dtype=float)

y_train_np = np.asarray(y_train_beta, dtype=float)
y_test_np = np.asarray(y_test_beta, dtype=float)

print(X_train_np.dtype)
print(y_train_np.dtype)
print(X_train_np.shape, y_train_np.shape)

In [ ]:
print(X_train_scaled.dtypes)
print(X_train_scaled.head())
print(type(X_train_np))
print(X_train_np.dtype)

In [ ]:
# Beta Bayesian Regression

with pm.Model() as beta_model:

    # Data containers
    X_data = pm.Data("x_data", X_train_np)
    y_data = pm.Data("y_data", y_train_np)

    # Priors
    intercept = pm.Normal("Intercept", mu=0, sigma=2)
    beta = pm.Normal("beta", mu=0, sigma=1, shape=n_features)

    # Precision / concentration parameter
    phi = pm.Gamma("phi", alpha=2, beta=0.1)

    # Linear predictor
    eta = intercept + pm.math.dot(X_data, beta)

    # Mean on (0,1)
    mu = pm.Deterministic("mu", pm.math.sigmoid(eta))

    # Beta parameters
    alpha_param = mu * phi
    beta_param = (1 - mu) * phi

    # Likelihood
    y_like = pm.Beta("y_like", alpha=alpha_param, beta=beta_param, observed=y_data)

    # Sampling
    trace_beta = pm.sample(
        1000,
        tune=1000,
        chains=2,
        target_accept=0.90,
        return_inferencedata=True,
        random_seed=42
    )

In [ ]:
# Posterior summary

summary_beta = az.summary(trace_beta, var_names=["Intercept", "beta", "phi"],round_to=4)
print(summary_beta)

In [ ]:
# Rename beta rows to match feature names

beta_rows = coef_summary.loc[coef_summary.index.str.startswith("beta[")].copy()
beta_rows["Feature"] = feature_names
beta_rows = beta_rows[["Feature", "mean", "sd", "hdi_3%", "hdi_97%", "r_hat"]]

print("\nFeature-level posterior summary:")
print(beta_rows)

In [ ]:
# Trace plots

az.plot_trace(trace_beta, var_names=["Intercept", "phi", "beta"])
plt.tight_layout()
plt.show()


In [ ]:
# Posterior prediction on test set

intercept_mean = coef_summary.loc["Intercept", "mean"]
phi_mean = coef_summary.loc["phi", "mean"]

beta_means = np.array([
    coef_summary.loc[f"beta[{i}]", "mean"] for i in range(n_features)
])

eta_test = intercept_mean + X_test_np @ beta_means
mu_test = 1 / (1 + np.exp(-eta_test))

# Convert predictions back to original PriceRatio scale
y_pred_ratio = 0.5 + mu_test * (1.5 - 0.5)

In [ ]:
# RMSE on original PriceRatio scale

rmse_beta = np.sqrt(np.mean((y_test.values - y_pred_ratio) ** 2))
print("\nBeta Bayesian Test RMSE:", round(rmse_beta, 4))

In [ ]:
# Predicted vs actual plot

plt.figure(figsize=(6, 5))
plt.scatter(y_test.values, y_pred_ratio, alpha=0.5)
plt.plot([0.5, 1.5], [0.5, 1.5], color="red")
plt.xlabel("Actual PriceRatio")
plt.ylabel("Predicted PriceRatio")
plt.title("Predicted vs Actual (Beta Bayesian)")
plt.show()

In [ ]:
# Residual plot

residuals_beta = y_test.values - y_pred_ratio

plt.figure(figsize=(6, 5))
plt.scatter(y_pred_ratio, residuals_beta, alpha=0.5)
plt.axhline(0, color="black")
plt.xlabel("Predicted PriceRatio")
plt.ylabel("Residuals (Actual - Predicted)")
plt.title("Residual Plot (Beta Bayesian)")
plt.show()

In [ ]:
print(y_train_np.min(), y_train_np.max())

In [ ]:
!jupyter nbconvert --to pdf '/content/drive/MyDrive/Colab Notebooks/Beta Distribution.ipynb'